# EVA AI - Layer Capture Model (Colab Version)

Этот notebook предназначен для запуска на Google Colab с GPU для работы с тяжёлыми моделями.

## Инструкция:
1. Загрузите этот notebook в Google Colab
2. Убедитесь, что выбран GPU runtime (Runtime → Change runtime type → GPU)
3. Загрузите файл `qwenlayermodel.pt` (8GB) в Colab
4. Запустите ячейки по порядку
5. Скачайте результаты обратно на локальную машину

In [ ]:
# Установка зависимостей
!pip install torch transformers openvino openvino-genai -q

In [ ]:
# Загрузка файла qwenlayermodel.pt
from google.colab import files
import os

print('Загрузите файл qwenlayermodel.pt (8GB)...')
# Раскомментируйте следующую строку для загрузки файла
# uploaded = files.upload()

# Или укажите путь, если файл уже загружен
checkpoint_path = '/content/qwenlayermodel.pt'
if not os.path.exists(checkpoint_path):
    checkpoint_path = list(uploaded.keys())[0] if 'uploaded' in locals() else None

print(f'Checkpoint path: {checkpoint_path}')
print(f'File exists: {os.path.exists(checkpoint_path) if checkpoint_path else False}')

In [ ]:
# Загрузка checkpoint и извлечение конфигурации
import torch
import json

print('Loading checkpoint...')
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
print(f'Checkpoint loaded. Keys: {list(checkpoint.keys())}')

# Извлечение конфигурации
config = checkpoint['config']
num_layers = checkpoint['num_layers']

print(f'Number of layers: {num_layers}')
print(f'Config type: {type(config)}')

# Сохранение конфигурации
config_dict = config.to_dict() if hasattr(config, 'to_dict') else config.__dict__
with open('/content/layer_capture_config.json', 'w') as f:
    json.dump(config_dict, f, indent=2)

print('Config saved to /content/layer_capture_config.json')

In [ ]:
# Создание упрощённой модели для захвата hidden states
from transformers import AutoModelForCausalLM, AutoConfig
import torch

print('Creating model from config...')

# Создаём конфиг
model_config = AutoConfig.from_pretrained(
    'RefalMachine/RuadaptQwen3-4B-Instruct',
    trust_remote_code=True
)

# Обновляем конфиг из checkpoint
config_dict = checkpoint['config'].to_dict()
for key, value in config_dict.items():
    if hasattr(model_config, key):
        setattr(model_config, key, value)

print(f'Config hidden_size: {model_config.hidden_size}')
print(f'Config num_layers: {model_config.num_hidden_layers}')

# Создаём модель
model = AutoModelForCausalLM.from_config(model_config, trust_remote_code=True)
print('Model created')

# Загружаем веса
model.load_state_dict(checkpoint['model_state_dict'], strict=False)
print('Weights loaded')

model.eval()
model = model.to('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Model moved to: {next(model.parameters()).device}')

In [ ]:
# Тест захвата hidden states
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Тестовый вход
test_input = torch.randint(0, 100, (1, 10)).to(device)
print(f'Input shape: {test_input.shape}')

# Захватываем hidden states
captured_states = {}

def make_hook(layer_num):
    def hook(module, input, output):
        if isinstance(output, tuple):
            captured_states[layer_num] = output[0].detach().cpu()
        else:
            captured_states[layer_num] = output.detach().cpu()
    return hook

# Регистрируем hooks
hooks = []
for i, layer in enumerate(model.model.layers):
    hook = layer.register_forward_hook(make_hook(i))
    hooks.append(hook)

print(f'Registered {len(hooks)} hooks')

# Forward pass
with torch.no_grad():
    outputs = model(test_input, output_hidden_states=True)

print(f'Captured {len(captured_states)} states')
for layer_num, state in captured_states.items():
    print(f'  Layer {layer_num}: shape={state.shape}')

# Удаляем hooks
for hook in hooks:
    hook.remove()

print('Done!')

In [ ]:
# Сохранение результатов
import torch
import json

# Сохраняем только важные веса (LoRA adapters, etc.)
print('Saving important weights...')

state_dict = model.state_dict()
important_keys = [k for k in state_dict.keys() if 'lora' in k.lower() or 'adapter' in k.lower()]

if important_keys:
    lora_dict = {k: state_dict[k] for k in important_keys}
    torch.save(lora_dict, '/content/lora_weights.pt')
    print(f'Saved {len(lora_dict)} LoRA weights to /content/lora_weights.pt')
else:
    print('No LoRA weights found')

# Сохраняем информацию о модели
model_info = {
    'num_layers': model_config.num_hidden_layers,
    'hidden_size': model_config.hidden_size,
    'num_heads': model_config.num_attention_heads,
    'intermediate_size': model_config.intermediate_size,
}

with open('/content/model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print('Model info saved to /content/model_info.json')

In [ ]:
# Скачивание результатов
from google.colab import files

print('Downloading results...')

files.download('/content/layer_capture_config.json')
files.download('/content/model_info.json')

# Если есть LoRA веса
import os
if os.path.exists('/content/lora_weights.pt'):
    files.download('/content/lora_weights.pt')

print('Done!')

## Результаты:

После выполнения этого notebook вы получите:
1. `layer_capture_config.json` - конфигурация модели
2. `model_info.json` - информация о размерностях
3. `lora_weights.pt` (опционально) - веса LoRA адаптеров

Эти файлы можно использовать на локальной машине для инициализации системы без загрузки 8GB модели.